In [5]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

In [6]:
class Transceiver:
    def __init__(self, name, temp_K, P_tx=None, G_tx=None, L_tx=None, 
                 EIRP=None, G_rx=0.0, L_rx=0.0, default_freq_MHz=None):
        """
        Transceiver specification for space communication link budget calculations.
        
        Parameters:
        - name (str): Transceiver identifier.
        - temp_K (float): System noise temperature in Kelvin.
        - P_tx (float, optional): Transmitter power in dBm.
        - G_tx (float, optional): Transmitter antenna gain in dBi.
        - L_tx (float, optional): Transmitter line/implementation loss in dB.
        - EIRP (float, optional): Explicit EIRP in dBm. Overrides P_tx, G_tx, L_tx if provided.
        - G_rx (float): Receiver antenna gain in dBi.
        - L_rx (float): Receiver line loss in dB.
        - default_freq_MHz (float, optional): Native operating frequency in MHz.
        """
        self.name = name
        self.temp_K = temp_K
        self.default_freq_MHz = default_freq_MHz
        
        # Calculate Noise Power Density (N0) in dBm/Hz: N0 = -198.6 + 10*log10(T)
        self.N0 = -198.60 + 10.0 * np.log10(self.temp_K)
        
        if EIRP is not None:
            self.EIRP = float(EIRP)
            self.P_tx = None
            self.G_tx = None
            self.L_tx = None
        else:
            if P_tx is None or G_tx is None or L_tx is None:
                raise ValueError("Must provide either 'EIRP' or all of ('P_tx', 'G_tx', 'L_tx').")
            self.P_tx = float(P_tx)
            self.G_tx = float(G_tx)
            self.L_tx = float(L_tx)
            self.EIRP = self.P_tx + self.G_tx - self.L_tx
            
        self.G_rx = float(G_rx)
        self.L_rx = float(L_rx)

    @property
    def net_rx_gain(self):
        """Returns net receiver gain (G_rx - L_rx) in dB."""
        return self.G_rx - self.L_rx

In [7]:
def plot_satellite_requirements(ground_station, candidate_spacecraft=None):
    """
    Plots the required Satellite Receiver Net Gain (Uplink) and Satellite Transmit EIRP (Downlink)
    needed to meet a target link margin given a specific Ground Station configuration.
    """
    # --- UI Widgets Initialization ---
    distance_slider = widgets.FloatSlider(
        value=400000.0, min=50000.0, max=1000000.0, step=10000.0,
        description='Distance (km):', continuous_update=False, style={'description_width': '140px'}
    )
    
    up_rate_slider = widgets.FloatLogSlider(
        value=10000.0, base=10, min=1, max=7, step=0.1,
        description='Uplink (bps):', continuous_update=False, style={'description_width': '140px'}
    )
    
    down_rate_slider = widgets.FloatLogSlider(
        value=100000.0, base=10, min=1, max=7, step=0.1,
        description='Downlink (bps):', continuous_update=False, style={'description_width': '140px'}
    )

    up_freq_slider = widgets.FloatSlider(
        value=2067.5, min=100.0, max=30000.0, step=25.0,
        description='Uplink Freq (MHz):', continuous_update=False, style={'description_width': '140px'}
    )

    down_freq_slider = widgets.FloatSlider(
        value=ground_station.default_freq_MHz if ground_station.default_freq_MHz else 2295.0,
        min=100.0, max=30000.0, step=25.0,
        description='Downlink Freq (MHz):', continuous_update=False, style={'description_width': '140px'}
    )

    sat_temp_slider = widgets.FloatSlider(
        value=450.0, min=50.0, max=1000.0, step=25.0,
        description='Sat Rx Temp (K):', continuous_update=False, style={'description_width': '140px'}
    )

    margin_slider = widgets.FloatSlider(
        value=3.0, min=0.0, max=10.0, step=0.5,
        description='Target Margin (dB):', continuous_update=False, style={'description_width': '140px'}
    )

    eb_n0_slider = widgets.FloatSlider(
        value=3.0, min=0.0, max=15.0, step=0.5,
        description='Req Eb/N0 (dB):', continuous_update=False, style={'description_width': '140px'}
    )

    prop_loss_slider = widgets.FloatSlider(
        value=0.0, min=0.0, max=10.0, step=0.5,
        description='Misc Loss (dB):', continuous_update=False, style={'description_width': '140px'}
    )

    def update_plots(distance_km, up_rate, down_rate, up_freq, down_freq, sat_temp, eb_n0, req_margin, prop_loss):
        d_range = np.linspace(50000, 1000000, 300)

        # --- 1. UPLINK: Satellite Receiver Net Gain Requirement ---
        # N0_sat = -198.60 + 10*log10(sat_temp)
        sat_N0 = -198.60 + 10.0 * np.log10(sat_temp)
        RS_sat = sat_N0 + 10 * np.log10(up_rate) + eb_n0
        
        L_FS_up_curve = 20 * np.log10(d_range) + 20 * np.log10(up_freq) + 32.44
        L_FS_up_curr = 20 * np.log10(distance_km) + 20 * np.log10(up_freq) + 32.44
        
        # Required Sat Net Rx Gain: G_net_sat = req_margin + RS_sat - EIRP_GS + L_FS + L_prop
        req_sat_net_rx_gain_curve = req_margin + RS_sat - ground_station.EIRP + L_FS_up_curve + prop_loss
        req_sat_net_rx_gain_curr = req_margin + RS_sat - ground_station.EIRP + L_FS_up_curr + prop_loss

        # --- 2. DOWNLINK: Satellite EIRP Requirement ---
        # Ground station receives signal
        RS_GS = ground_station.N0 + 10 * np.log10(down_rate) + eb_n0
        
        L_FS_down_curve = 20 * np.log10(d_range) + 20 * np.log10(down_freq) + 32.44
        L_FS_down_curr = 20 * np.log10(distance_km) + 20 * np.log10(down_freq) + 32.44
        
        # Required Sat EIRP: EIRP_sat = req_margin + RS_GS - G_net_GS + L_FS + L_prop
        req_sat_EIRP_curve = req_margin + RS_GS - ground_station.net_rx_gain + L_FS_down_curve + prop_loss
        req_sat_EIRP_curr = req_margin + RS_GS - ground_station.net_rx_gain + L_FS_down_curr + prop_loss

        # --- PLOTTING ---
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

        # Uplink Plot: Required Satellite Net Rx Gain
        ax1.plot(d_range / 1e3, req_sat_net_rx_gain_curve, 'b-', label='Min Req Satellite Net Rx Gain ($G_{rx}-L_{rx}$)', linewidth=2)
        ax1.axvline(distance_km / 1e3, color='red', linestyle=':', label=f'Target Distance ({distance_km:,.0f} km)')
        
        if candidate_spacecraft:
            cand_rx_gain = candidate_spacecraft.net_rx_gain
            ax1.axhline(cand_rx_gain, color='green', linestyle='--', label=f'Candidate Sat Gain ({cand_rx_gain:.1f} dBi)')

        ax1.set_title(f"Uplink Req: GS ({ground_station.name}) $\\rightarrow$ Satellite", fontsize=11, fontweight='bold')
        ax1.set_xlabel("Distance ($10^3$ km)")
        ax1.set_ylabel("Required Satellite Net Rx Gain ($G_{rx} - L_{rx}$) [dBi]")
        ax1.grid(True, linestyle=':', alpha=0.6)
        ax1.legend(loc='upper left', fontsize=9)

        text_up = (
            f"GS Transmit EIRP: {ground_station.EIRP:.1f} dBm\n"
            f"Freq: {up_freq:.1f} MHz | Data Rate: {up_rate/1e3:.1f} kbps\n"
            f"Path Loss ($L_{{FS}}$): {L_FS_up_curr:.2f} dB\n"
            f"Sat Rx Temp: {sat_temp:.0f} K ($N_0$: {sat_N0:.1f} dBm/Hz)\n"
            f"Sat Sensitivity ($RS$): {RS_sat:.2f} dBm\n"
            f"Required Sat Net Rx Gain: {req_sat_net_rx_gain_curr:+.2f} dBi"
        )
        ax1.text(0.03, 0.04, text_up, transform=ax1.transAxes, fontsize=8.5,
                 bbox=dict(boxstyle="round,pad=0.5", facecolor='white', edgecolor='navy', lw=1.2))

        # Downlink Plot: Required Satellite EIRP
        ax2.plot(d_range / 1e3, req_sat_EIRP_curve, 'm-', label='Min Req Satellite EIRP', linewidth=2)
        ax2.axvline(distance_km / 1e3, color='red', linestyle=':', label=f'Target Distance ({distance_km:,.0f} km)')
        
        if candidate_spacecraft:
            cand_eirp = candidate_spacecraft.EIRP
            ax2.axhline(cand_eirp, color='green', linestyle='--', label=f'Candidate Sat EIRP ({cand_eirp:.1f} dBm)')

        ax2.set_title(f"Downlink Req: Satellite $\\rightarrow$ GS ({ground_station.name})", fontsize=11, fontweight='bold')
        ax2.set_xlabel("Distance ($10^3$ km)")
        ax2.set_ylabel("Required Satellite Transmitter EIRP [dBm]")
        ax2.grid(True, linestyle=':', alpha=0.6)
        ax2.legend(loc='upper left', fontsize=9)

        text_down = (
            f"GS Net Receive Gain: {ground_station.net_rx_gain:.1f} dBi\n"
            f"GS Rx Temp: {ground_station.temp_K:.0f} K ($N_0$: {ground_station.N0:.1f} dBm/Hz)\n"
            f"Freq: {down_freq:.1f} MHz | Data Rate: {down_rate/1e3:.1f} kbps\n"
            f"Path Loss ($L_{{FS}}$): {L_FS_down_curr:.2f} dB\n"
            f"GS Sensitivity ($RS$): {RS_GS:.2f} dBm\n"
            f"Required Satellite EIRP: {req_sat_EIRP_curr:+.2f} dBm"
        )
        ax2.text(0.03, 0.04, text_down, transform=ax2.transAxes, fontsize=8.5,
                 bbox=dict(boxstyle="round,pad=0.5", facecolor='white', edgecolor='purple', lw=1.2))

        plt.tight_layout()
        plt.show()

    ui = widgets.VBox([
        widgets.HBox([distance_slider, prop_loss_slider, margin_slider]),
        widgets.HBox([up_rate_slider, down_rate_slider, eb_n0_slider]),
        widgets.HBox([up_freq_slider, down_freq_slider, sat_temp_slider])
    ])

    out = widgets.interactive_output(
        update_plots,
        {
            'distance_km': distance_slider,
            'up_rate': up_rate_slider,
            'down_rate': down_rate_slider,
            'up_freq': up_freq_slider,
            'down_freq': down_freq_slider,
            'sat_temp': sat_temp_slider,
            'eb_n0': eb_n0_slider,
            'req_margin': margin_slider,
            'prop_loss': prop_loss_slider
        }
    )

    display(ui, out)

In [9]:
dsn_34m = Transceiver(
    name="NASA DSN 34m Dish",
    temp_K=25.0,             # Background noise temp (~25 K)[cite: 1]
    EIRP=128.3,              # 128.3 dBm transmit EIRP[cite: 1]
    G_rx=56.74,              # 56.74 dBi receive gain[cite: 1]
    L_rx=0.0,
    default_freq_MHz=2295.0  # S-band downlink[cite: 1]
)

plot_satellite_requirements(dsn_34m)

Output()

In [10]:
cubesat_candidate = Transceiver(
    name="CubeSat Candidate",
    temp_K=450.0,            # 450 K noise temp[cite: 1]
    EIRP=30.0,               # 30 dBm EIRP[cite: 2]
    G_rx=0.0                 # 0 dBi Isotropic gain[cite: 1]
)

# Render interactive satellite design requirement curves for DSN
plot_satellite_requirements(dsn_34m, candidate_spacecraft=cubesat_candidate)

Output()

In [ ]:
nsn = Transceiver(
    name="NASA NSN",
    temp_K=150.0,             
    EIRP=56,              
    G_rx=42.76,  # Calculated from NSN G/T figure tranlsated to G_rx using G/T = G_rx - 10*log10(T)   
    L_rx=0.0,
    default_freq_MHz=2250.0  
)

plot_satellite_requirements(nsn)